In [0]:
order_reviews = spark.table("ecommerce_dev.silver.order_reviews")
order_reviews.printSchema()
print("order_reviews rows:", order_reviews.count())

root
 |-- review_id: string (nullable = false)
 |-- order_id: string (nullable = false)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)

order_reviews rows: 99224


In [0]:
from pyspark.sql.functions import *

orders = spark.table("ecommerce_dev.silver.orders")
dim_customer = spark.table("ecommerce_dev.gold.dim_customer")
dim_date = spark.table("ecommerce_dev.gold.dim_date")
order_reviews = spark.table("ecommerce_dev.silver.order_reviews")

fact_reviews = (
    order_reviews
    .join(orders.select("order_id", "customer_id", "order_purchase_timestamp"), "order_id", "left")
    .join(dim_customer.select("customer_key", "customer_id"), "customer_id", "left")
    .join(dim_date.select(col("date_key").alias("order_date_key"), col("full_date")),
          to_date("order_purchase_timestamp") == col("full_date"), "left")
    .select(
        "review_id",
        "order_id",
        "customer_key",
        "order_date_key",
        "review_score",
        "review_comment_title",
        "review_comment_message",
        "review_creation_date",
        "review_answer_timestamp"
    )
)

print("Fact rows:", fact_reviews.count())
print("Null customer_key:", fact_reviews.filter(col("customer_key").isNull()).count())
print("Null order_date_key:", fact_reviews.filter(col("order_date_key").isNull()).count())
print("Distinct (review_id, order_id):", fact_reviews.select("review_id","order_id").distinct().count())

Fact rows: 99224
Null customer_key: 0
Null order_date_key: 0
Distinct (review_id, order_id): 99224


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS ecommerce_dev.gold.fact_reviews (
    fact_review_key BIGINT GENERATED ALWAYS AS IDENTITY,
    review_id STRING,
    order_id STRING,
    customer_key BIGINT,
    order_date_key INT,
    review_score INT,
    review_comment_title STRING,
    review_comment_message STRING,
    review_creation_date TIMESTAMP,
    review_answer_timestamp TIMESTAMP
) USING DELTA
""")

fact_reviews.write.format("delta").mode("append").saveAsTable("ecommerce_dev.gold.fact_reviews")

spark.sql("ALTER TABLE ecommerce_dev.gold.fact_reviews ALTER COLUMN fact_review_key SET NOT NULL")
spark.sql("ALTER TABLE ecommerce_dev.gold.fact_reviews ADD CONSTRAINT pk_fact_reviews PRIMARY KEY (fact_review_key)")

spark.sql("""
COMMENT ON TABLE ecommerce_dev.gold.fact_reviews IS
'Review fact table at composite (review_id, order_id) grain, matching Silver PK — one review can map to multiple orders in split-seller checkouts.'
""")

DataFrame[]